In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import zipfile

Unzip all zip files. (Note: this only needs to be run once, it can be commented out after)

In [ ]:
# Define the function that will be called to unzip all files

def extract_zip_data(zip_file, subfolder):
   with zipfile.ZipFile(zip_file, 'r') as z:
       z.extractall(f"..\\data\\raw\\10Q_folder\\subfolders\\{subfolder}")

In [ ]:
# Make sure you have downloaded all the zip files manually from the SEC website and placed them in the zips path
# ONCE THIS IS RUN COMMENT EVERYTHING OUT
# THIS IS A ONE TIME EXERCISE TO EXTRACT THE ZIP FILES
for year in range(2010,2025):
   for q in range(1,5):
       subfolder = f"{year}q{q}"
       zip_file = f"..\\data\\raw\\10Q_folder\\zips\\{year}q{q}.zip"
       extract_zip_data(zip_file, subfolder)
       print(f"Extracted {year}q{q}.zip")


Combine all data 10Q NUM files into one

In [ ]:
# Define a function that reads in each SUB data frame and appends it to the master data frame
def read_sub_data(subfolder):
    data = pd.read_csv(f"..\\data\\raw\\10Q_folder\\subfolders\\{subfolder}\\sub.txt", sep='\t')
    # Append the data to the master data frame
    return data


In [ ]:
# ONCE THIS IS RUN WE CAN COMMENT EVERYTHING OUT Create a sub_master data frame to store all the data
# Append all sub data to the sub_master data frame

for year in range(2010,2025):
   for q in range(1,5):
       subfolder = f"{year}q{q}"
       sub_temp = read_sub_data(subfolder)
       if year == 2010 and q == 1:
           sub_master = sub_temp
       else:
           sub_master = sub_master._append(sub_temp)
       print(f"Appended {year}q{q}")

# Save the master data frame to a parquet file
sub_master.to_parquet("..\\data\\raw\\10Q_folder\\sub_master.parquet", index=False)
print("Master data frame saved to sub_master.parquet")

Appended 2010q1
Appended 2010q2
Appended 2010q3
Appended 2010q4
Appended 2011q1
Appended 2011q2
Appended 2011q3
Appended 2011q4
Appended 2012q1
Appended 2012q2
Appended 2012q3
Appended 2012q4
Appended 2013q1
Appended 2013q2
Appended 2013q3
Appended 2013q4
Appended 2014q1
Appended 2014q2
Appended 2014q3
Appended 2014q4
Appended 2015q1
Appended 2015q2
Appended 2015q3
Appended 2015q4
Appended 2016q1
Appended 2016q2
Appended 2016q3
Appended 2016q4
Appended 2017q1
Appended 2017q2
Appended 2017q3
Appended 2017q4
Appended 2018q1
Appended 2018q2
Appended 2018q3
Appended 2018q4
Appended 2019q1
Appended 2019q2
Appended 2019q3
Appended 2019q4
Appended 2020q1
Appended 2020q2
Appended 2020q3
Appended 2020q4
Appended 2021q1
Appended 2021q2
Appended 2021q3
Appended 2021q4
Appended 2022q1
Appended 2022q2
Appended 2022q3
Appended 2022q4
Appended 2023q1
Appended 2023q2
Appended 2023q3
Appended 2023q4
Appended 2024q1
Appended 2024q2
Appended 2024q3
Appended 2024q4
Master data frame saved to sub_master.cs

In [4]:
from pathlib import Path
from time import perf_counter

NUM_MASTER_PATH = Path(r"..\data\raw\10Q_folder\num_master.parquet")
NUM_SUBFOLDERS_PATH = Path(r"..\data\raw\10Q_folder\subfolders")

# Function to create labels for each SEC file folder
def sec_quarter_folders(start_year=2010, end_year=2024):
    return [
        f"{year}q{quarter}"
        for year in range(start_year, end_year + 1)
        for quarter in range(1, 5)
    ]

# Function to verify that each file exists
def read_num_data(subfolder, base_path=NUM_SUBFOLDERS_PATH):
    path = base_path / subfolder / "num.txt"
    if not path.exists():
        raise FileNotFoundError(f"Missing SEC NUM file: {path}")
    return pd.read_csv(path, sep="\t", low_memory=False)

# Read each num.txt file into a data frame and store the dfs in a list. 
# Perform one concatenation at the very end to avoid repeatedly appending millions of rows
def build_num_master(subfolders=None, base_path=NUM_SUBFOLDERS_PATH):
    subfolders = sec_quarter_folders() if subfolders is None else list(subfolders)
    frames = []
    started = perf_counter()

    for i, subfolder in enumerate(subfolders, start=1):
        quarter_started = perf_counter()
        frame = read_num_data(subfolder, base_path=base_path)
        frames.append(frame)
        print(
            f"Read {subfolder}: {len(frame):,} rows "
            f"in {perf_counter() - quarter_started:,.1f}s "
            f"({i}/{len(subfolders)})"
        )

    num_master = pd.concat(frames, ignore_index=True, copy=False)
    print(
        f"Built num_master: {len(num_master):,} rows x "
        f"{len(num_master.columns):,} columns "
        f"in {perf_counter() - started:,.1f}s"
    )
    return num_master

In [5]:
# Create a num_master data frame to store all the data
num_master = build_num_master()

# num_master.to_parquet(
#     NUM_MASTER_PATH,
#     engine="pyarrow",
#     compression="snappy",
#     index=False,
# )

# print(f"Master data frame saved to {NUM_MASTER_PATH}")

Read 2010q1: 194,741 rows in 0.3s (1/60)
Read 2010q2: 136,044 rows in 0.2s (2/60)
Read 2010q3: 480,612 rows in 0.6s (3/60)
Read 2010q4: 511,508 rows in 0.6s (4/60)
Read 2011q1: 690,483 rows in 0.9s (5/60)
Read 2011q2: 498,885 rows in 0.5s (6/60)
Read 2011q3: 2,025,154 rows in 2.3s (7/60)
Read 2011q4: 256,477 rows in 0.3s (8/60)
Read 2012q1: 2,699,630 rows in 3.8s (9/60)
Read 2012q2: 2,407,915 rows in 3.2s (10/60)
Read 2012q3: 2,816,079 rows in 3.5s (11/60)
Read 2012q4: 2,999,818 rows in 3.7s (12/60)
Read 2013q1: 3,156,166 rows in 4.0s (13/60)
Read 2013q2: 2,943,353 rows in 3.7s (14/60)
Read 2013q3: 3,081,119 rows in 3.8s (15/60)
Read 2013q4: 3,097,400 rows in 3.9s (16/60)
Read 2014q1: 3,483,361 rows in 4.5s (17/60)
Read 2014q2: 2,814,446 rows in 3.6s (18/60)
Read 2014q3: 2,936,590 rows in 3.7s (19/60)
Read 2014q4: 2,940,236 rows in 3.8s (20/60)
Read 2015q1: 3,322,479 rows in 4.4s (21/60)
Read 2015q2: 2,588,598 rows in 3.3s (22/60)
Read 2015q3: 2,780,346 rows in 3.7s (23/60)
Read 2015q4

In [6]:
num_master_small = num_master[num_master['segments'].isna()]

In [8]:
# Optionally run this if you want to save a copy at each point
num_master_small.to_parquet("..\\data\\raw\\10Q_folder\\num_master_small.parquet", engine="pyarrow", compression="snappy", index=False)

In [10]:
import pandas as pd
import fastparquet

pf = fastparquet.ParquetFile('..\\data\\raw\\10Q_folder\\num_master_small.parquet')
num_master_small = pf.to_pandas()

num_master_small.head(5)

,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote
0,0001047469-10-001364,ProfitLoss,us-gaap/2009,20081231,4,USD,<NA>,<NA>,8.015000e+08,None
1,0000950123-10-014582,NetChangeInAccruedInterestReceivablePayable,0000950123-10-014582,20071231,4,USD,<NA>,<NA>,7.800000e+05,None
2,0001193125-10-037944,WeightedAverageNumberOfSharesOutstandingBasic,us-gaap/2009,20071231,4,shares,<NA>,<NA>,1.815010e+08,None
3,0001047469-10-001028,WeightedAverageNumberOfSharesOutstandingBasic,us-gaap/2009,20081231,4,shares,<NA>,<NA>,1.826000e+08,None
4,0000821189-10-000011,PropertyPlantAndEquipmentGross,us-gaap/2009,20081231,0,USD,<NA>,<NA>,2.186152e+10,None


In [29]:
keep_rows = ['RevenueFromContractWithCustomerExcludingAssessedTax',
    'OperatingIncomeLoss',
    'NetIncomeLoss',
    'EarningsPerShareDiluted',
    'CostOfGoodsAndServicesSold',
    'ResearchAndDevelopmentExpense',
    'CashAndCashEquivalentsAtCarryingValue',
    'StockholdersEquity',
    'CommonStockSharesOutstanding',
    'DepreciationDepletionAndAmortization',
    'NetCashProvidedByUsedInOperatingActivities',
    'PaymentsToAcquirePropertyPlantAndEquipment',
    'Revenues',
    'OperatingCostAndExpense',
    'SellingGeneralAndAdministrativeExpense',
    'SellingGeneralAndAdministrativeExpenses',
    'SellingGeneralAdministrativeExpenses',
    'SellingGeneralAndAdministrativeExpenseExcludingDepreciationDepletionAndAmortization',
    'CostOfGoodsSold',
    'CostOfGoodsSoldExclusiveOfDepreciationAndAmortizationExpense',
    'OperatingCosts',
    'CashFlowsFromUsedInOperatingActivities',
    'CashFlowsFromUsedInOperations',
    'CapitalExpenditures',
    'DepreciationAndAmortization']

# Filter the rows where 'tags' column contains values in keep_rows
num_master_tiny = num_master_small[num_master_small['tag'].isin(keep_rows)]


In [ ]:
num_master_tiny.to_parquet("..\\data\\raw\\10Q_folder\\num_master_tiny_recon.parquet", engine="pyarrow", compression="snappy")
# Note: I did test this by loading the parquet and checking the dtypes against the original dataframe and they are the same

In [ ]:
# Optionally, test that the file can be read
import pandas as pd
num_master_tiny = pd.read_parquet("..\\data\\raw\\10Q_folder\\num_master_tiny_recon.parquet", engine="pyarrow")
num_master_tiny.head()

In [ ]:
sub_master = pd.read_parquet("..\\data\\raw\\10Q_folder\\sub_master.parquet", engine="pyarrow")
sub_master.to_parquet("..\\data\\raw\\10Q_folder\\sub_master.parquet", engine="pyarrow", compression="snappy")

In [31]:
# Merge the two data frames
num_sub_master = pd.merge(sub_master[['adsh','cik','name','sic','countryba','instance']], num_master_tiny, on='adsh')


In [ ]:
# !!! THIS IS ONE OF THE IMPORTANT FILES THAT IS USED IN THE PROD REPORT !!!
num_sub_master.to_parquet(".\\data\\clean\\num_sub_master_recon.parquet", engine="pyarrow", compression="snappy")

Go to 02_feature_engineering_pipeline.ipynb read in the parquet file and continue with data engineering